In [1]:
 # ============================================================
# NLP + K-MEANS CLUSTERING
# Dataset: 100kb.csv
# ============================================================

# ============================================================
# CELL 1 - INSTALL / IMPORT LIBRARIES
# ============================================================

# Run this cell if the libraries are not installed.
# !pip install pandas numpy scikit-learn matplotlib seaborn nltk

import pandas as pd
import numpy as np
import re
import warnings

import matplotlib.pyplot as plt
import seaborn as sns

import nltk

from nltk.corpus import stopwords

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

warnings.filterwarnings("ignore")

print("Libraries imported successfully.")


# ============================================================
# CELL 2 - DOWNLOAD NLTK DATA
# ============================================================

nltk.download("stopwords")

print("NLTK stopwords downloaded.")


# ============================================================
# CELL 3 - LOAD CSV DATASET
# ============================================================

file_path = "100kb.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully.")
print()
print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())


# ============================================================
# CELL 4 - DATASET INFORMATION
# ============================================================

print("========== DATASET INFORMATION ==========")

print("\nNumber of rows:")
print(df.shape[0])

print("\nNumber of columns:")
print(df.shape[1])

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())


# ============================================================
# CELL 5 - FIND TEXT COLUMNS
# ============================================================

print("========== POSSIBLE TEXT COLUMNS ==========")

for column in df.columns:

    if df[column].dtype == "object":

        print(
            f"\nColumn: {column}"
        )

        print(
            "Example:",
            str(df[column].dropna().iloc[0])[:200]
            if len(df[column].dropna()) > 0
            else "No data"
        )


# ============================================================
# CELL 6 - SELECT TEXT COLUMN
# ============================================================

# IMPORTANT:
# Change "text" to the actual text column in your CSV.
#
# Examples:
# text_column = "review"
# text_column = "description"
# text_column = "comment"
# text_column = "message"

text_column = "text"


# Check whether the selected column exists

if text_column not in df.columns:

    print("\nERROR:")
    print(f"Column '{text_column}' was not found.")

    print("\nAvailable columns are:")
    for column in df.columns:
        print("-", column)

    raise ValueError(
        f"Please change text_column = '{text_column}' "
        "to the correct text column."
    )


print("\nSelected text column:", text_column)


# ============================================================
# CELL 7 - REMOVE MISSING TEXT
# ============================================================

print("Rows before removing missing text:", len(df))

df = df.dropna(
    subset=[text_column]
).copy()

print("Rows after removing missing text:", len(df))


# ============================================================
# CELL 8 - CONVERT TEXT TO STRING
# ============================================================

df[text_column] = df[text_column].astype(str)

print("Text column converted to string.")


# ============================================================
# CELL 9 - REMOVE DUPLICATES
# ============================================================

before_duplicates = len(df)

df = df.drop_duplicates(
    subset=[text_column]
).copy()

after_duplicates = len(df)

print("Rows before duplicate removal:", before_duplicates)
print("Rows after duplicate removal:", after_duplicates)
print("Duplicates removed:", before_duplicates - after_duplicates)


# ============================================================
# CELL 10 - TEXT CLEANING FUNCTION
# ============================================================

stop_words = set(
    stopwords.words("english")
)


def clean_text(text):

    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(
        r"https?://\S+|www\.\S+",
        " ",
        text
    )

    # Remove email addresses
    text = re.sub(
        r"\S+@\S+",
        " ",
        text
    )

    # Remove HTML tags
    text = re.sub(
        r"<.*?>",
        " ",
        text
    )

    # Remove numbers
    text = re.sub(
        r"\d+",
        " ",
        text
    )

    # Remove punctuation
    text = re.sub(
        r"[^a-zA-Z\s]",
        " ",
        text
    )

    # Remove extra spaces
    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    # Remove stopwords
    words = []

    for word in text.split():

        if word not in stop_words:

            words.append(word)

    # Return cleaned sentence
    return " ".join(words)


print("Cleaning function created.")


# ============================================================
# CELL 11 - APPLY NLP CLEANING
# ============================================================

df["clean_text"] = df[text_column].apply(
    clean_text
)

print("Text cleaning completed.")


# ============================================================
# CELL 12 - VIEW ORIGINAL VS CLEANED TEXT
# ============================================================

print("========== ORIGINAL VS CLEANED TEXT ==========")

display(
    df[
        [text_column, "clean_text"]
    ].head(10)
)


# ============================================================
# CELL 13 - REMOVE EMPTY TEXT
# ============================================================

before_empty = len(df)

df = df[
    df["clean_text"].str.strip() != ""
].copy()

after_empty = len(df)

print("Rows before removing empty text:", before_empty)
print("Rows after removing empty text:", after_empty)

print("Empty rows removed:", before_empty - after_empty)


# ============================================================
# CELL 14 - TF-IDF VECTORIZATION
# ============================================================

vectorizer = TfidfVectorizer(

    # Maximum number of features
    max_features=5000,

    # Ignore words appearing in fewer than 2 documents
    min_df=2,

    # Ignore words appearing in more than 95% of documents
    max_df=0.95,

    # Use single words and two-word combinations
    ngram_range=(1, 2),

    # Remove English stopwords
    stop_words="english"
)


X = vectorizer.fit_transform(
    df["clean_text"]
)


print("TF-IDF completed.")

print(
    "TF-IDF matrix shape:",
    X.shape
)

print(
    "Number of documents:",
    X.shape[0]
)

print(
    "Number of features:",
    X.shape[1]
)


# ============================================================
# CELL 15 - DISPLAY TF-IDF FEATURES
# ============================================================

features = vectorizer.get_feature_names_out()

print("First 50 TF-IDF features:")

print(
    features[:50]
)


# ============================================================
# CELL 16 - FIND OPTIMAL K
# ============================================================

k_values = range(2, 11)

inertias = []

silhouette_scores = []


print("========== TESTING K VALUES ==========")

for k in k_values:

    print(
        f"\nTraining K-Means with K = {k}"
    )

    model = KMeans(

        n_clusters=k,

        random_state=42,

        n_init=10
    )

    labels = model.fit_predict(X)

    # Inertia
    inertias.append(
        model.inertia_
    )

    # Silhouette score
    score = silhouette_score(
        X,
        labels
    )

    silhouette_scores.append(
        score
    )

    print(
        f"Silhouette Score: {score:.4f}"
    )


# ============================================================
# CELL 17 - ELBOW GRAPH
# ============================================================

plt.figure(
    figsize=(10, 6)
)

plt.plot(
    k_values,
    inertias,
    marker="o"
)

plt.xlabel(
    "Number of Clusters (K)"
)

plt.ylabel(
    "Inertia"
)

plt.title(
    "Elbow Method for K-Means"
)

plt.xticks(
    list(k_values)
)

plt.grid(
    True
)

plt.show()


# ============================================================
# CELL 18 - SILHOUETTE GRAPH
# ============================================================

plt.figure(
    figsize=(10, 6)
)

plt.plot(
    k_values,
    silhouette_scores,
    marker="o"
)

plt.xlabel(
    "Number of Clusters (K)"
)

plt.ylabel(
    "Silhouette Score"
)

plt.title(
    "Silhouette Score for Different K Values"
)

plt.xticks(
    list(k_values)
)

plt.grid(
    True
)

plt.show()


# ============================================================
# CELL 19 - SELECT BEST K
# ============================================================

best_k_index = np.argmax(
    silhouette_scores
)

best_k = list(k_values)[
    best_k_index
]

best_score = silhouette_scores[
    best_k_index
]

print("========== BEST K ==========")

print(
    "Best number of clusters:",
    best_k
)

print(
    "Best silhouette score:",
    round(best_score, 4)
)


# ============================================================
# CELL 20 - TRAIN FINAL K-MEANS MODEL
# ============================================================

kmeans = KMeans(

    n_clusters=best_k,

    random_state=42,

    n_init=10
)


df["cluster"] = kmeans.fit_predict(
    X
)


print("Final K-Means model trained.")


# ============================================================
# CELL 21 - CLUSTER DISTRIBUTION
# ============================================================

cluster_counts = (
    df["cluster"]
    .value_counts()
    .sort_index()
)

print("========== CLUSTER DISTRIBUTION ==========")

print(
    cluster_counts
)


# ============================================================
# CELL 22 - CLUSTER DISTRIBUTION GRAPH
# ============================================================

plt.figure(
    figsize=(10, 6)
)

sns.barplot(
    x=cluster_counts.index,
    y=cluster_counts.values
)

plt.xlabel(
    "Cluster"
)

plt.ylabel(
    "Number of Documents"
)

plt.title(
    "Number of Documents in Each Cluster"
)

plt.show()


# ============================================================
# CELL 23 - FIND TOP WORDS IN EACH CLUSTER
# ============================================================

terms = vectorizer.get_feature_names_out()

cluster_centers = (
    kmeans.cluster_centers_
)


print("========== TOP WORDS BY CLUSTER ==========")


for cluster_id in range(best_k):

    # Get TF-IDF values for this cluster
    center = cluster_centers[
        cluster_id
    ]

    # Sort from highest to lowest
    top_indices = center.argsort()[
        ::-1
    ][:20]

    top_words = [
        terms[index]
        for index in top_indices
    ]

    print(
        f"\nCluster {cluster_id}"
    )

    print(
        ", ".join(top_words)
    )


# ============================================================
# CELL 24 - CREATE CLUSTER KEYWORDS DATAFRAME
# ============================================================

cluster_keywords = []


for cluster_id in range(best_k):

    center = cluster_centers[
        cluster_id
    ]

    top_indices = center.argsort()[
        ::-1
    ][:20]

    top_words = [
        terms[index]
        for index in top_indices
    ]

    cluster_keywords.append({

        "cluster": cluster_id,

        "top_keywords": ", ".join(
            top_words
        )
    })


keywords_df = pd.DataFrame(
    cluster_keywords
)


print(
    "Cluster keyword summary:"
)

display(
    keywords_df
)


# ============================================================
# CELL 25 - SHOW DOCUMENTS FROM EACH CLUSTER
# ============================================================

for cluster_id in range(best_k):

    print(
        "\n"
        + "=" * 80
    )

    print(
        f"CLUSTER {cluster_id}"
    )

    print(
        "=" * 80
    )

    cluster_data = df[
        df["cluster"] == cluster_id
    ]

    display(
        cluster_data[
            [text_column, "clean_text", "cluster"]
        ].head(10)
    )


# ============================================================
# CELL 26 - PCA DIMENSION REDUCTION
# ============================================================

print("Starting PCA...")

# PCA requires dense data.
#
# For a 100 KB dataset this should normally be manageable.
# If your dataset is much larger, use TruncatedSVD instead.

X_dense = X.toarray()


pca = PCA(
    n_components=2,
    random_state=42
)


X_pca = pca.fit_transform(
    X_dense
)


print(
    "PCA completed."
)

print(
    "Original shape:",
    X.shape
)

print(
    "PCA shape:",
    X_pca.shape
)


# ============================================================
# CELL 27 - ADD PCA COORDINATES TO DATAFRAME
# ============================================================

df["PCA_1"] = X_pca[:, 0]

df["PCA_2"] = X_pca[:, 1]


display(
    df[
        [
            text_column,
            "cluster",
            "PCA_1",
            "PCA_2"
        ]
    ].head()
)


# ============================================================
# CELL 28 - VISUALIZE K-MEANS CLUSTERS
# ============================================================

plt.figure(
    figsize=(12, 8)
)

scatter = plt.scatter(

    df["PCA_1"],

    df["PCA_2"],

    c=df["cluster"],

    cmap="viridis",

    alpha=0.7
)


plt.xlabel(
    "PCA Component 1"
)

plt.ylabel(
    "PCA Component 2"
)

plt.title(
    "K-Means NLP Clusters"
)

plt.colorbar(
    scatter,
    label="Cluster"
)

plt.grid(
    True
)

plt.show()


# ============================================================
# CELL 29 - CALCULATE FINAL SILHOUETTE SCORE
# ============================================================

final_silhouette = silhouette_score(
    X,
    df["cluster"]
)

print(
    "Final Silhouette Score:",
    round(
        final_silhouette,
        4
    )
)


# ============================================================
# CELL 30 - SHOW CLUSTER SUMMARY
# ============================================================

summary = (
    df
    .groupby("cluster")
    .size()
    .reset_index(
        name="document_count"
    )
)


summary["percentage"] = (
    summary["document_count"]
    / len(df)
    * 100
)


print("========== CLUSTER SUMMARY ==========")

display(
    summary
)


# ============================================================
# CELL 31 - COMBINE CLUSTER SUMMARY + KEYWORDS
# ============================================================

summary = summary.merge(
    keywords_df,
    on="cluster",
    how="left"
)


display(
    summary
)


# ============================================================
# CELL 32 - SAVE CLUSTERED DATA
# ============================================================

output_file = (
    "100kb_clustered.csv"
)


# Save the original text,
# cleaned text, cluster,
# and PCA coordinates.

df.to_csv(
    output_file,
    index=False
)


print(
    f"Clustered dataset saved as: {output_file}"
)


# ============================================================
# CELL 33 - SAVE CLUSTER SUMMARY
# ============================================================

summary_file = (
    "cluster_summary.csv"
)


summary.to_csv(
    summary_file,
    index=False
)


print(
    f"Cluster summary saved as: {summary_file}"
)


# ============================================================
# CELL 34 - FINAL RESULTS
# ============================================================

print()
print("=" * 60)
print("FINAL RESULTS")
print("=" * 60)

print(
    "Dataset:",
    file_path
)

print(
    "Documents:",
    len(df)
)

print(
    "TF-IDF features:",
    X.shape[1]
)

print(
    "Number of clusters:",
    best_k
)

print(
    "Silhouette score:",
    round(
        final_silhouette,
        4
    )
)

print()
print("Cluster sizes:")

print(
    df["cluster"]
    .value_counts()
    .sort_index()
)

print()
print("Output files:")

print(
    "- 100kb_clustered.csv"
)

print(
    "- cluster_summary.csv"
)

print()
print("Processing completed successfully.")

Libraries imported successfully.
NLTK stopwords downloaded.
Dataset loaded successfully.

Dataset shape: (1471, 6)

Column names:
['first_name', 'last_name', 'email', 'gender', 'ip_address', 'date']

First 5 rows:


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,first_name,last_name,email,gender,ip_address,date
0,Roobbie,Wilbore,rwilbore0@bandcamp.com,Female,82.243.17.211,1/7/2018
1,Aprilette,Dole,adole1@altervista.org,Female,12.132.123.75,3/27/2017
2,Phillipe,Winkworth,pwinkworth2@wufoo.com,Male,71.62.119.246,1/20/2018
3,Arvy,Lempke,alempke3@squidoo.com,Male,117.227.52.135,10/20/2017
4,Cody,Jakov,cjakov4@i2i.jp,Female,172.197.66.100,5/23/2017


========== DATASET INFORMATION ==========

Number of rows:
1471

Number of columns:
6

Data types:
first_name    str
last_name     str
email         str
gender        str
ip_address    str
date          str
dtype: object

Missing values:
first_name    0
last_name     0
email         0
gender        0
ip_address    0
date          0
dtype: int64

Duplicate rows:
0
========== POSSIBLE TEXT COLUMNS ==========

ERROR:
Column 'text' was not found.

Available columns are:
- first_name
- last_name
- email
- gender
- ip_address
- date


ValueError: Please change text_column = 'text' to the correct text column.